# Module 2: LULC Composition & Transition Matrix

**Task 2**: Quantify LULC composition and changes (2016, 2020, 2025) at state and district levels.

**Deliverables**:
- State-level composition tables + stacked bar / pie charts
- District-level composition heatmaps
- 9×9 transition matrices (3 period-pairs × 2 states)
- Sankey diagrams of major transitions
- Dominant transition per district choropleth
- Net change summary table

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from config import (
    STATES, YEARS, LULC_CLASSES, LULC_COLORS, NUM_CLASSES,
    GEE_EXPORTS_DIR, PROCESSED_DIR, FIGURES_DIR,
    TIER1_SCALE, PIXEL_AREA_KM2,
)
from scripts.lulc_utils import (
    parse_histogram, histogram_to_area_df,
    build_state_composition_table,
    parse_transition_histogram, compute_net_change,
    get_dominant_transition, compute_total_change_per_district,
    save_composition_csv, save_transition_matrix_csv,
)
from scripts.visualization import (
    plot_state_composition_bars, plot_state_composition_pie,
    plot_transition_heatmap, plot_net_change_bars,
    plot_district_choropleth,
)

print('Imports successful!')

## 2.1: Load State-Level Compositions

In [ ]:
# Load server-side composition results
with open(GEE_EXPORTS_DIR / 'state_compositions.json', 'r') as f:
    state_compositions = json.load(f)

# Build composition tables
composition_tables = {}

for state_key, state_cfg in STATES.items():
    table = build_state_composition_table(
        state_compositions[state_key],
        state_cfg['name'],
        YEARS,
        scale=TIER1_SCALE
    )
    composition_tables[state_key] = table
    
    # Save CSV
    save_composition_csv(
        table,
        PROCESSED_DIR / 'transition_matrices' / f'{state_key}_state_composition.csv'
    )
    
    # Display
    print(f'\n{state_cfg["name"]} State Composition:')
    display(table)

## 2.2: State-Level Composition Charts

In [ ]:
# Stacked bar charts and pie charts
for state_key, state_cfg in STATES.items():
    table = composition_tables[state_key]
    
    # Stacked bar chart
    plot_state_composition_bars(table, state_cfg['name'], YEARS)
    
    # Pie charts for each year
    for year in YEARS:
        plot_state_composition_pie(table, state_cfg['name'], year)

print('✅ All composition charts generated.')

## 2.3: District-Level Composition

Load the CSV files exported from GEE in Module 1.

In [ ]:
# Load district composition CSVs from GEE exports
# These files should be downloaded from Google Drive and placed in data/gee_exports/

district_comp_tables = {}

for state_key, state_cfg in STATES.items():
    district_comp_tables[state_key] = {}
    
    for year in YEARS:
        filepath = GEE_EXPORTS_DIR / f'{state_key}_district_composition_{year}.csv'
        
        if filepath.exists():
            df = pd.read_csv(filepath)
            district_comp_tables[state_key][year] = df
            print(f'Loaded: {filepath.name} ({len(df)} districts)')
        else:
            print(f'⚠️ Not found: {filepath.name}')
            print(f'   → Download from Google Drive and place in {GEE_EXPORTS_DIR}/')

## 2.4: District Composition Heatmaps

In [ ]:
# Heatmap of district-level LULC proportions
for state_key, state_cfg in STATES.items():
    for year in YEARS:
        if year not in district_comp_tables.get(state_key, {}):
            continue
        
        df = district_comp_tables[state_key][year]
        
        # Extract class columns and normalize to percentages
        class_cols = [c for c in LULC_CLASSES.values() if c in df.columns]
        if not class_cols:
            continue
        
        plot_data = df.set_index('district')[class_cols]
        plot_data_pct = plot_data.div(plot_data.sum(axis=1), axis=0) * 100
        
        fig, ax = plt.subplots(figsize=(14, max(8, len(df) * 0.4)))
        sns.heatmap(plot_data_pct, annot=True, fmt='.1f', cmap='YlGnBu',
                    ax=ax, cbar_kws={'label': 'Percentage (%)'})
        ax.set_title(f'District LULC Composition — {state_cfg["name"]} ({year})')
        ax.set_ylabel('District')
        plt.tight_layout()
        
        outpath = FIGURES_DIR / 'lulc_maps' / f'{state_key}_district_heatmap_{year}.png'
        outpath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=300, bbox_inches='tight')
        plt.close()
        print(f'📊 Saved: {outpath}')

## 2.5: State-Level Transition Matrices

In [ ]:
# Load transition data
with open(GEE_EXPORTS_DIR / 'state_transitions.json', 'r') as f:
    state_transitions = json.load(f)

period_pairs = [(2016, 2020), (2020, 2025), (2016, 2025)]

transition_matrices = {}
net_changes = {}

for state_key, state_cfg in STATES.items():
    transition_matrices[state_key] = {}
    net_changes[state_key] = {}
    
    for year_t1, year_t2 in period_pairs:
        label = f'{year_t1}_{year_t2}'
        period_label = f'{year_t1} → {year_t2}'
        
        # Parse transition matrix
        hist = state_transitions[state_key].get(label, {})
        matrix_df = parse_transition_histogram(hist, scale=TIER1_SCALE)
        transition_matrices[state_key][label] = matrix_df
        
        # Save CSV
        save_transition_matrix_csv(
            matrix_df,
            PROCESSED_DIR / 'transition_matrices' / f'{state_key}_transition_{label}.csv'
        )
        
        # Plot heatmap
        plot_transition_heatmap(matrix_df, state_cfg['name'], period_label)
        
        # Net change
        net_df = compute_net_change(matrix_df)
        net_changes[state_key][label] = net_df
        plot_net_change_bars(net_df, state_cfg['name'], period_label)
        
        # Print dominant transition
        dominant = get_dominant_transition(matrix_df)
        print(f'{state_cfg["name"]} {period_label}: '
              f'Dominant = {dominant["label"]} ({dominant["area_km2"]:.1f} km²)')

print('\n✅ All transition matrices generated.')

## 2.6: Sankey Diagrams

Using plotly for interactive Sankey flow diagrams.

In [ ]:
import plotly.graph_objects as go

def make_sankey(matrix_df, state_name, period_label, top_n=15):
    """Create Sankey diagram from transition matrix."""
    class_names = list(matrix_df.index)
    n = len(class_names)
    
    # Node labels: source classes (left) and target classes (right)
    labels = [f'{c} ({period_label.split("→")[0].strip()})' for c in class_names] + \
             [f'{c} ({period_label.split("→")[1].strip()})' for c in class_names]
    
    colors_node = [LULC_COLORS[i] for i in range(n)] * 2
    
    # Flows (exclude diagonal = no-change)
    sources, targets, values, colors_link = [], [], [], []
    
    for i in range(n):
        for j in range(n):
            val = matrix_df.iloc[i, j]
            if val > 0:
                sources.append(i)
                targets.append(n + j)
                values.append(val)
                colors_link.append(LULC_COLORS[i] + '80')  # semi-transparent
    
    # Keep only top N flows for clarity
    if len(values) > top_n:
        sorted_idx = np.argsort(values)[::-1][:top_n]
        sources = [sources[i] for i in sorted_idx]
        targets = [targets[i] for i in sorted_idx]
        values = [values[i] for i in sorted_idx]
        colors_link = [colors_link[i] for i in sorted_idx]
    
    fig = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=labels, color=colors_node),
        link=dict(source=sources, target=targets, value=values, color=colors_link)
    ))
    fig.update_layout(
        title=f'LULC Transitions — {state_name} ({period_label})',
        font_size=11, width=900, height=600
    )
    
    # Save as HTML (interactive)
    outdir = FIGURES_DIR / 'transition_sankey'
    outdir.mkdir(parents=True, exist_ok=True)
    safe_label = period_label.replace(' → ', '_to_')
    filepath = outdir / f'{state_name.lower()}_sankey_{safe_label}.html'
    fig.write_html(str(filepath))
    print(f'📊 Saved: {filepath}')
    
    return fig

# Generate Sankey for all state-period combinations
for state_key, state_cfg in STATES.items():
    for year_t1, year_t2 in period_pairs:
        label = f'{year_t1}_{year_t2}'
        period_label = f'{year_t1} → {year_t2}'
        
        matrix_df = transition_matrices[state_key][label]
        fig = make_sankey(matrix_df, state_cfg['name'], period_label)
        fig.show()

## 2.7: District-Level Dominant Transitions & Ranking

In [ ]:
# Load district transition CSVs and compute dominant transitions
district_rankings = {}

for state_key, state_cfg in STATES.items():
    # Try loading district transition data
    # The 2016→2025 period gives the overall change
    filepath = GEE_EXPORTS_DIR / f'{state_key}_district_transitions_2016_2025.csv'
    
    if not filepath.exists():
        print(f'⚠️ {filepath.name} not found — download from Google Drive')
        continue
    
    df = pd.read_csv(filepath)
    
    # Parse transition histograms per district
    district_transitions = {}
    for _, row in df.iterrows():
        district_name = row.get('ADM2_NAME', row.get('district', f'District_{_}'))
        hist_str = row.get('transition_histogram', '{}')
        try:
            hist = json.loads(hist_str) if isinstance(hist_str, str) else hist_str
        except json.JSONDecodeError:
            hist = {}
        district_transitions[district_name] = hist
    
    # Compute rankings
    ranking_df = compute_total_change_per_district(district_transitions, scale=TIER1_SCALE)
    district_rankings[state_key] = ranking_df
    
    # Save
    save_composition_csv(
        ranking_df,
        PROCESSED_DIR / 'transition_matrices' / f'{state_key}_district_change_ranking.csv'
    )
    
    print(f'\n{state_cfg["name"]} — Districts ranked by total change:')
    display(ranking_df.head(10))
    
    # Identify highest-change district for Module 3
    top_district = ranking_df.iloc[0]['district']
    print(f'\n🏆 Highest-change district: {top_district}')

## 2.8: District Choropleth — Dominant Transition

In [ ]:
# Load district boundaries and join with ranking data
from config import BOUNDARIES_DIR

for state_key, state_cfg in STATES.items():
    boundary_file = BOUNDARIES_DIR / f'{state_key}_district_boundaries.geojson'
    
    if not boundary_file.exists():
        print(f'⚠️ Boundary file not found: {boundary_file}')
        continue
    
    if state_key not in district_rankings:
        continue
    
    gdf = gpd.read_file(boundary_file)
    ranking_df = district_rankings[state_key]
    
    # Merge
    merged = gdf.merge(
        ranking_df,
        left_on='ADM2_NAME', right_on='district',
        how='left'
    )
    
    # Choropleth of total change
    plot_district_choropleth(
        merged, 'change_pct',
        f'LULC Change Intensity (%) — {state_cfg["name"]} (2016→2025)',
        cmap='YlOrRd',
        legend_title='Change (%)',
        state_name=state_key
    )

print('\n✅ Module 2 complete. Proceed to Module 3 for highest-change district analysis.')

---

## Summary of Results

This notebook produced:
1. ✅ State-level LULC composition tables (2 states × 3 years)
2. ✅ Stacked bar charts and pie charts
3. ✅ District-level composition heatmaps
4. ✅ 9×9 transition matrices with heatmaps (3 period-pairs × 2 states)
5. ✅ Sankey diagrams (interactive HTML)
6. ✅ Net change bar charts
7. ✅ District ranking by total change
8. ✅ Dominant transition choropleth maps

**Key output for Module 3**: The highest-change district has been identified.

**Next**: `03_district_change.ipynb`